<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Data Carpentry · Python profesional</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Soluciones — Funciones reutilizables, Type Hints y Docstrings**

En este laboratorio pasaremos de bloques de código que simplemente funcionan a **funciones reutilizables, comprensibles, verificables y documentadas**.

Seguiremos la metodología del curso:

> **ver el problema → diagnosticarlo → corregirlo → validar la corrección → documentar la solución**

## Objetivos de aprendizaje

Al finalizar deberías poder:

1. Identificar código que debería convertirse en función.
2. Diseñar funciones con una responsabilidad clara.
3. Diferenciar parámetros obligatorios y opcionales.
4. Usar valores por defecto de forma segura.
5. Evitar efectos secundarios innecesarios.
6. Utilizar `type hints`.
7. Entender que los tipos no validan automáticamente.
8. Incorporar validación explícita.
9. Escribir docstrings profesionales.
10. Comprobar casos límite.
11. Crear una pequeña API funcional reusable.
12. Trasladar funciones estables a módulos `.py`.

## Cómo trabajar

**Solución:** incluye la misma parte guiada y doce ejercicios resueltos con pruebas de contrato, resultados y efectos secundarios. Las respuestas abiertas admiten alternativas justificadas.

**Entorno:** Python, NumPy, Pandas y Matplotlib. Trabaja desde la carpeta de la sesión. Se crean demos en `demos_funciones/`; las soluciones añaden `practicas_funciones/`. Los datos son sintéticos. Los type hints no sustituyen la validación.

**Selección de actividades:** la parte de funciones corresponde al bloque de modularización; type hints, contratos y docstrings permiten continuar en la siguiente sesión del curso. El docente seleccionará los ejercicios de aula y las ampliaciones.


## 1. Preparación del entorno

In [ ]:
from pathlib import Path
from typing import Iterable, Optional, Union, Sequence
import inspect
import sys
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Entorno preparado correctamente.")

## 2. Detectar código que debería convertirse en función

In [ ]:
clientes_1 = pd.DataFrame({
    "ingresos": [30000, 45000, 52000],
    "gasto": [12000, 16000, 22000]
})

clientes_2 = pd.DataFrame({
    "ingresos": [28000, 60000, 39000],
    "gasto": [15000, 18000, 17000]
})

clientes_1["ratio_gasto"] = clientes_1["gasto"] / clientes_1["ingresos"]
clientes_2["ratio_gasto"] = clientes_2["gasto"] / clientes_2["ingresos"]

clientes_1, clientes_2

### Diagnóstico

La misma transformación aparece en dos lugares. Si cambia la regla, tendremos que actualizar ambos bloques.

Eso es una señal clara de que conviene extraer una función.

## 3. Primera función reusable

In [ ]:
def añadir_ratio_gasto(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()
    salida["ratio_gasto"] = salida["gasto"] / salida["ingresos"]
    return salida

clientes_1_ref = añadir_ratio_gasto(clientes_1[["ingresos", "gasto"]])
clientes_1_ref

### Interpretación

La transformación ya tiene un nombre, una entrada y una salida. Además, usamos `.copy()` para no modificar silenciosamente el objeto original.

## 4. Efectos secundarios: primero vemos el problema

In [ ]:
def añadir_ratio_mal(datos):
    datos["ratio_gasto"] = datos["gasto"] / datos["ingresos"]

demo = pd.DataFrame({
    "ingresos": [100, 200],
    "gasto": [50, 80]
})

print("ANTES")
display(demo)

resultado = añadir_ratio_mal(demo)

print("DESPUÉS")
display(demo)

print("Retorno:", resultado)

### Inconsistencia observada

La función ha modificado el DataFrame original y devuelve `None`.

En transformación de datos este patrón suele resultar poco claro. Preferiremos funciones que devuelvan explícitamente un nuevo objeto.

## 5. Corrección del efecto secundario

In [ ]:
def añadir_ratio_seguro(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()
    salida["ratio_gasto"] = salida["gasto"] / salida["ingresos"]
    return salida

demo_original = pd.DataFrame({
    "ingresos": [100, 200],
    "gasto": [50, 80]
})

demo_transformado = añadir_ratio_seguro(demo_original)

print("ORIGINAL")
display(demo_original)

print("TRANSFORMADO")
display(demo_transformado)

## 6. Parámetros obligatorios y opcionales

In [ ]:
def clasificar_ingreso(
    ingreso: float,
    umbral: float = 40000
) -> str:
    if ingreso >= umbral:
        return "alto"
    return "normal"

print(clasificar_ingreso(50000))
print(clasificar_ingreso(50000, umbral=60000))

`ingreso` es obligatorio. `umbral` es opcional y configurable.

Los valores por defecto son útiles cuando existe un comportamiento habitual, pero no deberían ocultar decisiones críticas del negocio.

## 7. Parámetro mutable por defecto: provocar el error

In [ ]:
def añadir_etiqueta_mal(etiqueta, etiquetas=[]):
    etiquetas.append(etiqueta)
    return etiquetas

print(añadir_etiqueta_mal("A"))
print(añadir_etiqueta_mal("B"))
print(añadir_etiqueta_mal("C"))

### ¿Qué ha ocurrido?

La lista por defecto se reutiliza entre llamadas. Se ha creado un estado persistente inesperado.

## 7.1 Corregir el parámetro mutable

In [ ]:
def añadir_etiqueta(
    etiqueta: str,
    etiquetas: Optional[list[str]] = None
) -> list[str]:

    if etiquetas is None:
        etiquetas = []

    etiquetas.append(etiqueta)
    return etiquetas

print(añadir_etiqueta("A"))
print(añadir_etiqueta("B"))
print(añadir_etiqueta("C"))

## 8. Función con demasiadas responsabilidades

In [ ]:
def analizar_clientes_mal(datos):
    datos = datos.copy()
    datos = datos.drop_duplicates()
    datos["ratio_gasto"] = datos["gasto"] / datos["ingresos"]
    datos["alto_consumo"] = datos["ratio_gasto"] > 0.60

    resumen = (
        datos.groupby("segmento")
        .agg(
            clientes=("ingresos", "size"),
            ratio_medio=("ratio_gasto", "mean")
        )
        .reset_index()
    )

    print(resumen)
    return resumen

### Diagnóstico

Esta función limpia, crea features, clasifica, agrega e imprime. Eso dificulta reutilizar o probar cada parte por separado.

## 9. Separación de responsabilidades

In [ ]:
def eliminar_duplicados(datos: pd.DataFrame) -> pd.DataFrame:
    return datos.drop_duplicates().copy()


def calcular_ratio_gasto(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()
    salida["ratio_gasto"] = salida["gasto"] / salida["ingresos"]
    return salida


def clasificar_alto_consumo(
    datos: pd.DataFrame,
    umbral: float = 0.60
) -> pd.DataFrame:
    salida = datos.copy()
    salida["alto_consumo"] = salida["ratio_gasto"] > umbral
    return salida


def resumir_segmentos(datos: pd.DataFrame) -> pd.DataFrame:
    return (
        datos.groupby("segmento")
        .agg(
            clientes=("ingresos", "size"),
            ratio_medio=("ratio_gasto", "mean")
        )
        .reset_index()
    )

## 10. Dataset de trabajo

In [ ]:
n = 1000

clientes = pd.DataFrame({
    "cliente_id": np.arange(1, n + 1),
    "segmento": rng.choice(["A", "B", "C"], n),
    "ingresos": rng.normal(35000, 12000, n).clip(8000),
    "gasto": rng.normal(16000, 7000, n).clip(500)
})

clientes.head()

## 11. Type hints: primero una función ambigua

In [ ]:
def media_valores_sin_tipos(valores):
    return sum(valores) / len(valores)

media_valores_sin_tipos([1, 2, 3])

La función funciona, pero la firma no comunica claramente qué clase de entrada espera ni qué devuelve.

## 12. Añadir type hints

In [ ]:
def media_valores(
    valores: Iterable[float]
) -> float:
    valores = list(valores)

    if not valores:
        raise ValueError(
            "No se puede calcular la media de una colección vacía."
        )

    return sum(valores) / len(valores)

media_valores([1, 2, 3, 4])

### Interpretación

`Iterable[float]` indica una colección iterable de valores numéricos. La función declara además que devuelve `float`.

## 13. Los type hints NO validan automáticamente

In [ ]:
def duplicar(
    valor: int
) -> int:
    return valor * 2

print(duplicar(3))
print(duplicar("hola"))

### Inconsistencia

Python ejecuta la segunda llamada aunque la anotación indique `int`.

Por tanto:

**type hint ≠ validación en runtime**

## 14. Validación explícita

In [ ]:
def calcular_descuento(
    importe: float,
    porcentaje: float
) -> float:

    if not isinstance(importe, (int, float)):
        raise TypeError("importe debe ser numérico.")

    if not isinstance(porcentaje, (int, float)):
        raise TypeError("porcentaje debe ser numérico.")

    if importe < 0:
        raise ValueError("importe no puede ser negativo.")

    if not 0 <= porcentaje <= 1:
        raise ValueError("porcentaje debe estar entre 0 y 1.")

    return importe * (1 - porcentaje)

calcular_descuento(100, 0.20)

## 14.1 Ver los errores antes de corregir los datos

In [ ]:
casos_invalidos = [
    (-100, 0.2),
    (100, 1.5),
    ("100", 0.2)
]

for importe, porcentaje in casos_invalidos:
    try:
        print(calcular_descuento(importe, porcentaje))
    except Exception as error:
        print(type(error).__name__, "->", error)

### Interpretación

La función detiene el proceso con errores explícitos y relacionados con el dominio.

## 15. Tipos frecuentes en funciones profesionales

In [ ]:
def obtener_nombre(cliente: dict[str, str]) -> str:
    return cliente["nombre"]


def rango(valores: Sequence[float]) -> tuple[float, float]:
    return min(valores), max(valores)


def buscar_cliente(cliente_id: int) -> Optional[str]:
    if cliente_id == 1:
        return "Ana"
    return None


def normalizar_id(identificador: Union[int, str]) -> str:
    return str(identificador).strip()


print(obtener_nombre({"nombre": "Marc"}))
print(rango([2, 5, 8]))
print(buscar_cliente(3))
print(normalizar_id("  ABC  "))

Lectura rápida:

- `dict[str, str]`: diccionario de strings.
- `Sequence[float]`: secuencia numérica.
- `tuple[float, float]`: tupla de dos floats.
- `Optional[str]`: string o `None`.
- `Union[int, str]`: entero o string.

## 16. Type hints en DataFrames y su limitación

In [ ]:
def filtrar_ingresos(
    datos: pd.DataFrame,
    minimo: float
) -> pd.DataFrame:
    return datos.loc[
        datos["ingresos"] >= minimo
    ].copy()

filtrar_ingresos(clientes, 40000).head()

`pd.DataFrame` indica el tipo de objeto, pero no garantiza que exista una columna llamada `ingresos`.

## 16.1 Mostrar la inconsistencia

In [ ]:
clientes_incorrectos = pd.DataFrame({
    "income": [20000, 40000]
})

try:
    filtrar_ingresos(clientes_incorrectos, 30000)
except Exception as error:
    print(type(error).__name__, "->", error)

## 16.2 Corregir mediante validación

In [ ]:
def filtrar_ingresos_seguro(
    datos: pd.DataFrame,
    minimo: float
) -> pd.DataFrame:

    if "ingresos" not in datos.columns:
        raise ValueError(
            "El DataFrame debe contener la columna 'ingresos'."
        )

    if minimo < 0:
        raise ValueError(
            "El mínimo de ingresos no puede ser negativo."
        )

    return datos.loc[
        datos["ingresos"] >= minimo
    ].copy()


try:
    filtrar_ingresos_seguro(clientes_incorrectos, 30000)
except Exception as error:
    print(type(error).__name__, "->", error)

## 17. Docstrings: primero una función mal documentada

In [ ]:
def calcular_ticket_medio_mal(datos, columna):
    # Devuelve una media.
    return datos[columna].mean()

El comentario no explica el contrato completo: parámetros, retorno, errores o ejemplo de uso.

## 18. Docstring profesional estilo NumPy

In [ ]:
def calcular_ticket_medio(
    datos: pd.DataFrame,
    columna: str = "importe"
) -> float:
    """Calcula el valor medio de una columna numérica.

    Parameters
    ----------
    datos : pd.DataFrame
        DataFrame que contiene la variable de interés.
    columna : str, default="importe"
        Nombre de la columna numérica usada en el cálculo.

    Returns
    -------
    float
        Media aritmética de la columna seleccionada.

    Raises
    ------
    KeyError
        Si la columna indicada no existe.
    TypeError
        Si la columna no es numérica.

    Examples
    --------
    >>> df = pd.DataFrame({"importe": [10, 20, 30]})
    >>> calcular_ticket_medio(df)
    20.0
    """
    if columna not in datos.columns:
        raise KeyError(
            f"No existe la columna {columna!r}."
        )

    if not pd.api.types.is_numeric_dtype(datos[columna]):
        raise TypeError(
            f"La columna {columna!r} debe ser numérica."
        )

    return float(datos[columna].mean())

## 18.1 Consultar la documentación

In [ ]:
help(calcular_ticket_medio)

## 18.2 Inspeccionar automáticamente firma y docstring

In [ ]:
print(inspect.signature(calcular_ticket_medio))
print()
print(inspect.getdoc(calcular_ticket_medio))

### Interpretación

La documentación y las anotaciones pueden ser utilizadas por IDEs, generadores de documentación y herramientas de análisis.

## 19. Docstring existente pero insuficiente

In [ ]:
def funcion_poco_documentada(x):
    """Calcula cosas."""
    return x * 2

Tener una docstring no implica que sea útil. La documentación debe aportar información que no sea obvia simplemente leyendo el nombre.

# 20. Crear una pequeña API funcional

In [ ]:
def validar_clientes(datos: pd.DataFrame) -> None:
    """Valida estructura y reglas básicas del dataset."""

    requeridas = {
        "cliente_id",
        "segmento",
        "ingresos",
        "gasto"
    }

    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if datos["ingresos"].isna().any():
        raise ValueError("Existen ingresos nulos.")

    if (datos["ingresos"] <= 0).any():
        raise ValueError("Los ingresos deben ser positivos.")

In [ ]:
def preparar_clientes(datos: pd.DataFrame) -> pd.DataFrame:
    """Añade variables derivadas al dataset."""
    salida = datos.copy()
    salida["ratio_gasto"] = salida["gasto"] / salida["ingresos"]
    return salida

In [ ]:
def resumir_clientes(datos: pd.DataFrame) -> pd.DataFrame:
    """Resume indicadores por segmento."""
    return (
        datos.groupby("segmento")
        .agg(
            clientes=("cliente_id", "size"),
            ingreso_medio=("ingresos", "mean"),
            gasto_medio=("gasto", "mean"),
            ratio_medio=("ratio_gasto", "mean")
        )
        .reset_index()
    )

## 20.1 Ejecutar la API

In [ ]:
validar_clientes(clientes)

clientes_preparados = preparar_clientes(clientes)
resumen = resumir_clientes(clientes_preparados)

resumen

## 20.2 Visualizar el resultado

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    resumen["segmento"],
    resumen["ratio_medio"]
)

ax.set_title("Ratio medio de gasto por segmento")
ax.set_xlabel("Segmento")
ax.set_ylabel("Ratio gasto / ingresos")

plt.show()

### Interpretación

El gráfico consume un resultado analítico ya preparado. La visualización no necesita conocer cómo se validó o transformó el dataset.

# 21. Casos límite: probar antes de confiar

In [ ]:
casos = {
    "correcto": clientes.head(10).copy(),

    "sin_ingresos": clientes.head(10).drop(
        columns=["ingresos"]
    ),

    "ingreso_negativo": clientes.head(10).assign(
        ingresos=lambda x:
            x["ingresos"].mask(
                x.index == x.index[0],
                -100
            )
    ),

    "ingreso_nulo": clientes.head(10).assign(
        ingresos=lambda x:
            x["ingresos"].mask(
                x.index == x.index[0],
                np.nan
            )
    )
}

In [ ]:
for nombre, datos in casos.items():

    print("\nCASO:", nombre)

    try:
        validar_clientes(datos)
        print("Validación correcta")
    except Exception as error:
        print(type(error).__name__, "->", error)

## 21.1 Convertir las pruebas en una tabla

In [ ]:
resultados_pruebas = []

for nombre, datos in casos.items():

    try:
        validar_clientes(datos)
        estado = "OK"
        mensaje = ""

    except Exception as error:
        estado = "ERROR"
        mensaje = str(error)

    resultados_pruebas.append({
        "caso": nombre,
        "estado": estado,
        "mensaje": mensaje
    })

tabla_pruebas = pd.DataFrame(resultados_pruebas)
tabla_pruebas

## 21.2 Visualizar los resultados

In [ ]:
conteo_estados = tabla_pruebas["estado"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    conteo_estados.index,
    conteo_estados.values
)

ax.set_title("Resultado de las pruebas manuales")
ax.set_ylabel("Número de casos")

plt.show()

### Interpretación

Más adelante sustituiremos este mecanismo por pruebas automatizadas con `pytest`, pero aquí interesa entender primero la lógica de un caso de prueba.

# 22. Crear automáticamente un módulo reusable

In [ ]:
import tempfile
base_demos = Path("demos_funciones")
base_demos.mkdir(exist_ok=True)
raiz = Path(tempfile.mkdtemp(prefix="proyecto_", dir=base_demos)).resolve()
src = raiz / "src" / "clientes_utils"
src.mkdir(parents=True, exist_ok=True)
(src / "__init__.py").write_text("", encoding="utf-8")
print("Área de demostración:", raiz)


In [ ]:
validation_module = """
import pandas as pd

def validar_clientes(datos: pd.DataFrame) -> None:
    \"\"\"Valida las columnas mínimas del dataset.\"\"\"

    requeridas = {
        "cliente_id",
        "segmento",
        "ingresos",
        "gasto"
    }

    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if datos["ingresos"].isna().any():
        raise ValueError("Existen ingresos nulos.")

    if (datos["ingresos"] <= 0).any():
        raise ValueError("Los ingresos deben ser positivos.")
"""

(src / "validation.py").write_text(
    textwrap.dedent(validation_module),
    encoding="utf-8"
)

print("validation.py creado.")

## 22.1 Inspeccionar el módulo generado

In [ ]:
print(
    (src / "validation.py").read_text(
        encoding="utf-8"
    )
)

## 22.2 Importar la función desde el módulo

In [ ]:
ruta_src = str((raiz / "src").resolve())

if ruta_src not in sys.path:
    sys.path.insert(0, ruta_src)

from clientes_utils.validation import validar_clientes as validar_clientes_modulo

validar_clientes_modulo(clientes.head(20))

print("Import y validación correctos.")

### Interpretación

La función ya no está encerrada en el notebook. Puede ser consumida por notebooks, scripts, tests o APIs.

# 23. Comparar firmas de funciones

In [ ]:
funciones = [
    añadir_ratio_gasto,
    calcular_descuento,
    filtrar_ingresos_seguro,
    calcular_ticket_medio,
    validar_clientes,
    preparar_clientes,
    resumir_clientes
]

for funcion in funciones:
    print(
        funcion.__name__,
        "->",
        inspect.signature(funcion)
    )

### Interpretación

Una buena firma permite comprender buena parte del contrato sin abrir el cuerpo de la función.

# 24. Mini-práctica guiada — API analítica de ventas

In [ ]:
ventas = pd.DataFrame({
    "venta_id": np.arange(1, 501),
    "producto": rng.choice(["A", "B", "C", "D"], 500),
    "unidades": rng.integers(1, 10, 500),
    "precio": rng.uniform(5, 120, 500)
})

ventas.head()

## Paso 1 — Validar

In [ ]:
def validar_ventas(datos: pd.DataFrame) -> None:

    requeridas = {
        "venta_id",
        "producto",
        "unidades",
        "precio"
    }

    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise ValueError(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if (datos["unidades"] <= 0).any():
        raise ValueError(
            "Las unidades deben ser positivas."
        )

    if (datos["precio"] < 0).any():
        raise ValueError(
            "El precio no puede ser negativo."
        )

## Paso 2 — Transformar

In [ ]:
def preparar_ventas(datos: pd.DataFrame) -> pd.DataFrame:
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    return salida

## Paso 3 — Clasificar

In [ ]:
def clasificar_ventas(
    datos: pd.DataFrame,
    umbral: float = 500
) -> pd.DataFrame:

    salida = datos.copy()
    salida["venta_alta"] = salida["importe"] >= umbral
    return salida

## Paso 4 — Resumir

In [ ]:
def resumir_ventas(datos: pd.DataFrame) -> pd.DataFrame:
    return (
        datos.groupby("producto")
        .agg(
            ventas=("venta_id", "size"),
            facturacion=("importe", "sum"),
            ticket_medio=("importe", "mean"),
            pct_altas=("venta_alta", "mean")
        )
        .reset_index()
    )

## Paso 5 — Componer el pipeline

In [ ]:
def ejecutar_pipeline_ventas(
    datos: pd.DataFrame,
    umbral_venta_alta: float = 500
) -> tuple[pd.DataFrame, pd.DataFrame]:

    validar_ventas(datos)

    preparados = preparar_ventas(datos)

    clasificados = clasificar_ventas(
        preparados,
        umbral=umbral_venta_alta
    )

    resumen = resumir_ventas(clasificados)

    return clasificados, resumen


ventas_finales, resumen_ventas = (
    ejecutar_pipeline_ventas(
        ventas,
        umbral_venta_alta=500
    )
)

resumen_ventas

## Paso 6 — Visualizar e interpretar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    resumen_ventas["producto"],
    resumen_ventas["facturacion"]
)

ax.set_title("Facturación por producto")
ax.set_xlabel("Producto")
ax.set_ylabel("Facturación")

plt.show()

### Interpretación

El pipeline está compuesto por funciones pequeñas y sustituibles. Podemos cambiar validación, feature engineering o agregación sin reescribirlo completamente.

# 25. Checklist de diseño de funciones

- [ ] ¿El nombre comunica intención?
- [ ] ¿Tiene una responsabilidad principal?
- [ ] ¿Las entradas son claras?
- [ ] ¿La salida es explícita?
- [ ] ¿Evita efectos secundarios innecesarios?
- [ ] ¿Los valores por defecto son seguros?
- [ ] ¿Los tipos están anotados?
- [ ] ¿Se validan reglas críticas?
- [ ] ¿Los errores son informativos?
- [ ] ¿La docstring explica parámetros, retorno y errores?
- [ ] ¿Se han probado casos límite?
- [ ] ¿La función podría vivir fuera del notebook?

# 26. Resumen del laboratorio

Hemos trabajado tres pilares:

### Funciones reutilizables
Convierten lógica repetida en unidades con intención y contrato.

### Type hints
Mejoran comunicación y análisis estático, pero no sustituyen validación.

### Docstrings
Permiten utilizar una función sin leer toda su implementación.

Juntos, estos elementos hacen que el código sea mucho más mantenible y reutilizable.

# 27. Ejercicios finales


## Ejercicio 1 — Extraer funciones

Refactoriza tres bloques que calculan el ratio gasto/ingresos. Utiliza una función con una responsabilidad y verifica que los tres resultados y las entradas se conservan.


In [ ]:
grupos_practica = [pd.DataFrame({"ingresos": [30000, 45000], "gasto": [12000, 16000]}),
                    pd.DataFrame({"ingresos": [52000, 28000], "gasto": [18000, 11000]}),
                    pd.DataFrame({"ingresos": [40000, 60000], "gasto": [15000, 22000]})]
originales_grupos = [grupo.copy(deep=True) for grupo in grupos_practica]
referencia_1 = grupos_practica[0].copy()
referencia_1["ratio_gasto"] = referencia_1["gasto"] / referencia_1["ingresos"]
referencia_2 = grupos_practica[1].copy()
referencia_2["ratio_gasto"] = referencia_2["gasto"] / referencia_2["ingresos"]
referencia_3 = grupos_practica[2].copy()
referencia_3["ratio_gasto"] = referencia_3["gasto"] / referencia_3["ingresos"]

def ratio_practica(datos: pd.DataFrame) -> pd.DataFrame:
    """Añade el ratio a una copia; presupone ingresos positivos y finitos."""
    salida = datos.copy()
    salida["ratio_gasto"] = salida["gasto"] / salida["ingresos"]
    return salida

for grupo, original, referencia in zip(grupos_practica, originales_grupos, [referencia_1, referencia_2, referencia_3]):
    pd.testing.assert_frame_equal(ratio_practica(grupo), referencia)
    pd.testing.assert_frame_equal(grupo, original)
print("Tres resultados equivalentes y entradas conservadas.")


### Solución razonada

La regla compartida tiene un único lugar de implementación. La validación es una responsabilidad que puede realizarse antes de esta función; documentar sus supuestos permite saber cuándo es seguro usarla.


## Ejercicio 2 — Parámetros

Diseña importe_neto = unidades × precio × (1 − descuento), con descuento opcional igual a cero. Define el contrato y prueba descuento 0 y 1. En esta práctica las unidades y el precio deben ser positivos.


In [ ]:
from numbers import Real
import math

def importe_neto_practica(unidades: float, precio: float, descuento: float = 0.0) -> float:
    """Calcula el importe neto de una operación.

    Parameters
    ----------
    unidades : float
        Cantidad estrictamente positiva y finita.
    precio : float
        Precio unitario estrictamente positivo y finito.
    descuento : float, default 0.0
        Fracción de descuento entre 0 y 1, ambos incluidos.

    Returns
    -------
    float
        Unidades por precio por uno menos el descuento.

    Raises
    ------
    TypeError
        Si un argumento no es un número real o es booleano.
    ValueError
        Si un argumento no es finito o está fuera del rango permitido.

    Examples
    --------
    >>> importe_neto_practica(2, 10, 0.1)
    18.0
    """
    for nombre, valor in [("unidades", unidades), ("precio", precio), ("descuento", descuento)]:
        if isinstance(valor, bool) or not isinstance(valor, Real):
            raise TypeError(f"{nombre} debe ser un número real y no booleano.")
        if not math.isfinite(valor):
            raise ValueError(f"{nombre} debe ser finito.")
    if unidades <= 0 or precio <= 0:
        raise ValueError("Unidades y precio deben ser positivos.")
    if not 0 <= descuento <= 1:
        raise ValueError("Descuento fuera de [0, 1].")
    return float(unidades * precio * (1 - descuento))

assert importe_neto_practica(2, 10) == 20
assert importe_neto_practica(2, 10, 0.1) == 18
assert importe_neto_practica(2, 10, 1) == 0


### Solución razonada

Los parámetros expresan cantidades y configuración de la regla. Los type hints describen la firma, pero las comprobaciones dentro del cuerpo hacen cumplir el contrato. Los booleanos se rechazan explícitamente porque bool es una subclase de int.


## Ejercicio 3 — Parámetro mutable

Provoca la acumulación de elementos al usar una lista como valor por defecto. Corrige con None. Define además si la función modificará una lista proporcionada por el llamador.


In [ ]:
def etiquetas_mal_practica(etiqueta, etiquetas=[]):
    etiquetas.append(etiqueta)
    return etiquetas

primera_lista = etiquetas_mal_practica("A")
segunda_lista = etiquetas_mal_practica("B")
assert primera_lista is segunda_lista
assert primera_lista == ["A", "B"]

def etiquetas_practica(etiqueta: str, etiquetas: Optional[list[str]] = None) -> list[str]:
    salida = [] if etiquetas is None else list(etiquetas)
    salida.append(etiqueta)
    return salida

assert etiquetas_practica("A") == ["A"]
assert etiquetas_practica("B") == ["B"]
lista_entrada = ["inicio"]
assert etiquetas_practica("fin", lista_entrada) == ["inicio", "fin"]
assert lista_entrada == ["inicio"]
print("Default compartido reproducido y corregido; entrada conservada.")


### Solución razonada

El valor por defecto se crea al definir la función. None permite crear una nueva lista por llamada. Copiar una lista explícita también evita mutación del llamador; son dos decisiones distintas. La copia es superficial y basta para este ejemplo con cadenas.


## Ejercicio 4 — Type hints

Escribe firmas con list, dict, tuple, Optional y Union. Incluye una función con cada tipo y explica qué significan. Usa argumentos válidos sin confundir anotación con validación en ejecución.


In [ ]:
def sumar_lista(valores: list[float]) -> float:
    return float(sum(valores))

def convertir_registro(nombre: str, importe: float) -> dict[str, Union[str, float]]:
    return {"nombre": nombre, "importe": importe}

def extremos(valores: list[float]) -> tuple[float, float]:
    if not valores:
        raise ValueError("Se requiere al menos un valor.")
    return min(valores), max(valores)

def buscar_valor(registro: dict[str, float], clave: str) -> Optional[float]:
    return registro.get(clave)

def normalizar_numero(valor: Union[int, float]) -> float:
    return float(valor)

assert sumar_lista([1.0, 2.0]) == 3.0
assert convertir_registro("A", 10.0)["importe"] == 10.0
assert extremos([1.0, 3.0, 2.0]) == (1.0, 3.0)
assert buscar_valor({"A": 10.0}, "B") is None
assert normalizar_numero(2) == 2.0
for funcion in [sumar_lista, convertir_registro, extremos, buscar_valor, normalizar_numero]:
    print(funcion.__name__, inspect.signature(funcion))


### Solución razonada

list describe elementos, dict claves y valores, tuple una salida de longitud fija. Optional[T] significa T o None, no que el argumento tenga automáticamente un valor por defecto. Union admite varios tipos posibles. La ejecución no impone estas anotaciones por sí misma.


## Ejercicio 5 — Validación

Valida columnas, valores positivos y categorías permitidas. Contrato: productos A/B/C; canales web/tienda/partner; tabla no vacía; descuento en [0, 1]. Rechaza nulos y valores numéricos no finitos.


In [ ]:
datos_practica = pd.DataFrame({
    "producto": ["A", "B", "A", "C", "B", "C"],
    "unidades": [2, 3, 1, 4, 2, 1],
    "precio": [10.0, 20.0, 15.0, 8.0, 30.0, 50.0],
    "descuento": [0.0, 0.1, 0.2, 1.0, 0.0, 0.2],
    "canal": ["web", "tienda", "partner", "web", "tienda", "partner"],
})
def validar_practica(datos: pd.DataFrame) -> None:
    """Valida el contrato de ventas de la práctica.

    Parameters
    ----------
    datos : pandas.DataFrame
        Tabla no vacía con producto, unidades, precio, descuento y canal.

    Returns
    -------
    None
        No transforma la entrada.

    Raises
    ------
    TypeError
        Si datos no es un DataFrame.
    ValueError
        Si faltan columnas, hay nulos, tipos inválidos o valores fuera de rango.

    Examples
    --------
    >>> validar_practica(pd.DataFrame({"producto": ["A"], "unidades": [2],
    ...     "precio": [10.0], "descuento": [0.1], "canal": ["web"]}))
    """
    if not isinstance(datos, pd.DataFrame):
        raise TypeError("datos debe ser un DataFrame.")
    requeridas = {"producto", "unidades", "precio", "descuento", "canal"}
    faltantes = requeridas - set(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas: {sorted(faltantes)}")
    if datos.empty:
        raise ValueError("Se requiere al menos una operación.")
    if datos[list(requeridas)].isna().any().any():
        raise ValueError("Hay nulos en columnas obligatorias.")
    for columna in ["unidades", "precio", "descuento"]:
        if not pd.api.types.is_numeric_dtype(datos[columna]) or pd.api.types.is_bool_dtype(datos[columna]):
            raise ValueError(f"{columna} debe ser numérica y no booleana.")
        if not np.isfinite(datos[columna].to_numpy(dtype=float)).all():
            raise ValueError(f"{columna} debe contener valores finitos.")
    if (datos[["unidades", "precio"]] <= 0).any().any():
        raise ValueError("Unidades y precio deben ser positivos.")
    if not datos["descuento"].between(0, 1).all():
        raise ValueError("Descuento fuera de [0, 1].")
    if not datos["canal"].isin(["web", "tienda", "partner"]).all():
        raise ValueError("Canal no permitido.")
    if not datos["producto"].isin(["A", "B", "C"]).all():
        raise ValueError("Producto no permitido.")

validar_practica(datos_practica)
print("Contrato válido.")


### Solución razonada

Primero se comprueba el esquema para no producir KeyError al acceder a una columna inexistente. Rechazar tablas vacías es una política de esta práctica, no una obligación universal. Un pipeline de producción debe decidir y documentar cómo tratar una entrada vacía.


## Ejercicio 6 — Docstrings

Documenta importe_neto_practica, validar_practica y preparar_practica con Parameters, Returns, Raises y Examples. Consulta su documentación y verifica los ejemplos.


In [ ]:
def preparar_practica(datos: pd.DataFrame) -> pd.DataFrame:
    """Añade importes sin modificar la tabla de entrada.

    Parameters
    ----------
    datos : pandas.DataFrame
        Tabla validada con unidades, precio y descuento.

    Returns
    -------
    pandas.DataFrame
        Copia con importe_bruto e importe_neto.

    Raises
    ------
    KeyError
        Si se omiten columnas del contrato; validar antes de llamar.

    Examples
    --------
    >>> ejemplo = pd.DataFrame({"unidades": [2], "precio": [10.0], "descuento": [0.1]})
    >>> preparar_practica(ejemplo)["importe_neto"].iloc[0]
    18.0
    """
    salida = datos.copy()
    salida["importe_bruto"] = salida["unidades"] * salida["precio"]
    salida["importe_neto"] = salida["importe_bruto"] * (1 - salida["descuento"])
    return salida

for funcion in [importe_neto_practica, validar_practica, preparar_practica]:
    doc = inspect.getdoc(funcion)
    assert doc is not None
    assert all(seccion in doc for seccion in ["Parameters", "Returns", "Raises", "Examples"])
    print(funcion.__name__, inspect.signature(funcion))
    print(doc)
assert importe_neto_practica(2, 10, 0.1) == 18.0
validar_practica(pd.DataFrame({"producto": ["A"], "unidades": [2], "precio": [10.0], "descuento": [0.1], "canal": ["web"]}))
ejemplo_doc = pd.DataFrame({"unidades": [2], "precio": [10.0], "descuento": [0.1]})
assert preparar_practica(ejemplo_doc)["importe_neto"].iloc[0] == 18.0


### Solución razonada

La documentación explica rangos y efectos secundarios, no solo la fórmula. preparar_practica asume validación previa: su docstring comunica esa precondición. Los ejemplos de las tres funciones se ejecutan y se comprueba su resultado.


## Ejercicio 7 — Casos límite

Prueba entradas correctas, vacías, con columna inexistente, negativo y nulo. Amplía con descuento inválido, categoría no permitida, texto e infinito. Define el error esperado para cada caso y comprueba que ocurre.


In [ ]:
casos_practica = {"correcto": datos_practica.copy(),
                  "vacio": datos_practica.iloc[:0].copy(),
                  "sin_precio": datos_practica.drop(columns="precio"),
                  "precio_negativo": datos_practica.assign(precio=-1),
                  "precio_nulo": datos_practica.assign(precio=np.nan),
                  "precio_infinito": datos_practica.assign(precio=np.inf),
                  "precio_texto": datos_practica.assign(precio="texto"),
                  "descuento_invalido": datos_practica.assign(descuento=1.1),
                  "canal_invalido": datos_practica.assign(canal="otro"),
                  "producto_invalido": datos_practica.assign(producto="Z")}
resultados_casos = []
for nombre, datos_caso in casos_practica.items():
    error_esperado = None if nombre == "correcto" else ValueError
    try:
        validar_practica(datos_caso)
    except ValueError as error:
        if error_esperado is None:
            raise
        resultados_casos.append({"caso": nombre, "resultado": "rechazo esperado", "detalle": str(error)})
    else:
        if error_esperado is not None:
            raise AssertionError(f"Se aceptó el caso inválido: {nombre}")
        resultados_casos.append({"caso": nombre, "resultado": "aceptación esperada", "detalle": ""})
print(pd.DataFrame(resultados_casos))
# Los escalares también deben cumplir su contrato.
for argumentos, tipo_error in [((0, 10, 0), ValueError), ((2, -10, 0), ValueError),
                               ((2, 10, 1.1), ValueError), ((2, float("nan"), 0), ValueError),
                               ((True, 10, 0), TypeError), ((2, "10", 0), TypeError)]:
    try:
        importe_neto_practica(*argumentos)
    except tipo_error:
        pass
    else:
        raise AssertionError(f"Se aceptaron argumentos inválidos: {argumentos}")


### Solución razonada

Una prueba debe distinguir un rechazo esperado de un error accidental. Capturar indiscriminadamente Exception y marcar éxito podría ocultar defectos. Aquí el caso correcto debe pasar y todos los casos inválidos deben generar el tipo previsto.


## Ejercicio 8 — API funcional

Compón validar, preparar, clasificar y resumir. Clasifica como alta solo cuando el neto sea mayor que el umbral. Cada función debe tener una responsabilidad; no debe dibujar ni exportar.


In [ ]:
def clasificar_practica(datos: pd.DataFrame, umbral: float = 50.0) -> pd.DataFrame:
    """Añade venta_alta cuando importe_neto supera estrictamente el umbral."""
    if isinstance(umbral, bool) or not isinstance(umbral, Real):
        raise TypeError("El umbral debe ser real y no booleano.")
    if not math.isfinite(umbral) or umbral < 0:
        raise ValueError("El umbral debe ser finito y no negativo.")
    salida = datos.copy()
    salida["venta_alta"] = salida["importe_neto"] > umbral
    return salida

def resumir_practica(datos: pd.DataFrame) -> pd.DataFrame:
    """Resume operaciones, importe neto y proporción alta por producto."""
    return datos.groupby("producto", as_index=False).agg(
        operaciones=("producto", "size"), importe_neto=("importe_neto", "sum"),
        pct_alta=("venta_alta", "mean"))

validar_practica(datos_practica)
preparados_practica = preparar_practica(datos_practica)
clasificados_practica = clasificar_practica(preparados_practica)
resumen_practica = resumir_practica(clasificados_practica)
np.testing.assert_allclose(resumen_practica["importe_neto"].sum(), preparados_practica["importe_neto"].sum())
limites_practica = clasificar_practica(pd.DataFrame({"importe_neto": [49.0, 50.0, 51.0]}))
assert limites_practica["venta_alta"].tolist() == [False, False, True]
print(resumen_practica)


### Solución razonada

La frontera del umbral se prueba de forma explícita. La preparación no decide una política de clasificación, y el resumen no modifica las filas originales. El resultado analítico se presenta desde el notebook.


## Ejercicio 9 — Módulo .py

Genera e importa un módulo con importe_neto_practica. Utiliza una carpeta nueva de práctica y comprueba que el módulo produce el mismo resultado.


In [ ]:
import tempfile
import importlib
base_practicas = Path("practicas_funciones")
base_practicas.mkdir(exist_ok=True)
raiz_practica = Path(tempfile.mkdtemp(prefix="funciones_", dir=base_practicas)).resolve()
fuente_importes = "from numbers import Real\nimport math\n\n" + inspect.getsource(importe_neto_practica)
ruta_modulo = raiz_practica / "importes_practica.py"
ruta_modulo.write_text(fuente_importes, encoding="utf-8")
compile(fuente_importes, str(ruta_modulo), "exec")
sys.modules.pop("importes_practica", None)
sys.path.insert(0, str(raiz_practica))
importlib.invalidate_caches()
import importes_practica
assert importes_practica.importe_neto_practica(2, 10, 0.1) == importe_neto_practica(2, 10, 0.1)
assert inspect.getdoc(importes_practica.importe_neto_practica) == inspect.getdoc(importe_neto_practica)
print("Módulo importado y docstring conservada:", ruta_modulo)


### Solución razonada

La función necesita sus imports también dentro del módulo: el namespace del notebook no se transfiere automáticamente. Se añade la ruta para la demostración; una librería instalable utilizaría packaging e instalación editable.


## Ejercicio 10 — Efectos secundarios

Muestra una función que modifica su DataFrame de entrada y otra que devuelve una copia. Comprueba que la versión segura no altera columnas ni valores del original.


In [ ]:
def preparar_con_mutacion(datos: pd.DataFrame) -> None:
    datos["importe_neto"] = datos["unidades"] * datos["precio"] * (1 - datos["descuento"])

demo_mutacion = datos_practica.copy()
assert preparar_con_mutacion(demo_mutacion) is None
assert "importe_neto" in demo_mutacion.columns
entrada_segura = datos_practica.copy(deep=True)
copia_segura = entrada_segura.copy(deep=True)
salida_segura = preparar_practica(entrada_segura)
pd.testing.assert_frame_equal(entrada_segura, copia_segura)
assert salida_segura is not entrada_segura
assert "importe_neto" not in entrada_segura
assert "importe_neto" in salida_segura
print("Mutación demostrada; versión segura conserva la entrada.")


### Solución razonada

Devolver None no significa que una función no haya hecho nada: puede haber mutado el argumento. Aquí la copia de DataFrame protege la entrada de las asignaciones realizadas. Una copia no es necesariamente profunda respecto de objetos Python anidados en columnas object.


## Ejercicio 11 — Pipeline

Construye un pipeline con validación, preparación, clasificación y resumen. Devuelve detalle y resumen; compara con un cálculo de referencia independiente y prueba el rechazo de entradas inválidas.


In [ ]:
def pipeline_practica(datos: pd.DataFrame, umbral: float = 50.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Valida y compone las etapas, devolviendo detalle y resumen."""
    validar_practica(datos)
    preparados = preparar_practica(datos)
    clasificados = clasificar_practica(preparados, umbral)
    return clasificados, resumir_practica(clasificados)

entrada_pipeline_original = datos_practica.copy(deep=True)
detalle_pipeline, resumen_pipeline = pipeline_practica(datos_practica)
referencia_pipeline = datos_practica.copy()
referencia_pipeline["importe_bruto"] = referencia_pipeline["unidades"] * referencia_pipeline["precio"]
referencia_pipeline["importe_neto"] = [u * p * (1 - d) for u, p, d in zip(
    referencia_pipeline["unidades"], referencia_pipeline["precio"], referencia_pipeline["descuento"])]
referencia_pipeline["venta_alta"] = referencia_pipeline["importe_neto"] > 50
pd.testing.assert_frame_equal(detalle_pipeline, referencia_pipeline)
referencia_resumen = referencia_pipeline.groupby("producto", as_index=False).agg(
    operaciones=("producto", "size"), importe_neto=("importe_neto", "sum"), pct_alta=("venta_alta", "mean"))
pd.testing.assert_frame_equal(resumen_pipeline, referencia_resumen)
pd.testing.assert_frame_equal(datos_practica, entrada_pipeline_original)
try:
    pipeline_practica(datos_practica.assign(unidades=0))
except ValueError as error:
    print("Entrada inválida rechazada:", error)
else:
    raise AssertionError("El pipeline aceptó unidades cero.")
print("Detalle, resumen y ausencia de mutación comprobados.")


### Solución razonada

La orquestación fija el orden y mantiene independientes las etapas. La referencia calcula el neto mediante una comprensión y compara resultados, en lugar de limitarse a volver a llamar a las mismas funciones.


## Ejercicio 12 — Reto final

Crea una librería con validation.py, features.py y analysis.py; añade pipeline.py y API pública. Conserva type hints y docstrings, prueba casos límite y genera una visualización final con 1.500 ventas reproducibles.


In [ ]:
raiz_libreria = raiz_practica / "libreria"
paquete_libreria = raiz_libreria / "src" / "ventas_funciones"
paquete_libreria.mkdir(parents=True, exist_ok=True)
fuentes_libreria = {
    "validation.py": "import numpy as np\nimport pandas as pd\n\n" + inspect.getsource(validar_practica),
    "features.py": "import pandas as pd\nfrom numbers import Real\nimport math\n\n" + inspect.getsource(preparar_practica) + "\n" + inspect.getsource(clasificar_practica),
    "analysis.py": "import pandas as pd\n\n" + inspect.getsource(resumir_practica),
    "pipeline.py": "from .validation import validar_practica\nfrom .features import preparar_practica, clasificar_practica\nfrom .analysis import resumir_practica\nimport pandas as pd\n\n" + inspect.getsource(pipeline_practica),
    "__init__.py": "from .pipeline import pipeline_practica\nfrom .validation import validar_practica\n__all__ = ['pipeline_practica', 'validar_practica']\n",
}
for nombre, fuente in fuentes_libreria.items():
    compile(fuente, nombre, "exec")
    (paquete_libreria / nombre).write_text(fuente, encoding="utf-8")
for nombre in list(sys.modules):
    if nombre == "ventas_funciones" or nombre.startswith("ventas_funciones."):
        sys.modules.pop(nombre)
sys.path.insert(0, str((raiz_libreria / "src").resolve()))
importlib.invalidate_caches()
from ventas_funciones import pipeline_practica as pipeline_libreria
from ventas_funciones import validar_practica as validar_libreria
rng_reto = np.random.default_rng(2026)
datos_reto = pd.DataFrame({
    "producto": rng_reto.choice(["A", "B", "C"], 1500),
    "unidades": rng_reto.integers(1, 10, 1500),
    "precio": rng_reto.uniform(5, 100, 1500),
    "descuento": rng_reto.choice([0.0, 0.1, 0.2], 1500),
    "canal": rng_reto.choice(["web", "tienda", "partner"], 1500),
})
copia_reto = datos_reto.copy(deep=True)
detalle_reto, resumen_reto = pipeline_libreria(datos_reto, umbral=200)
detalle_local, resumen_local = pipeline_practica(datos_reto, umbral=200)
pd.testing.assert_frame_equal(detalle_reto, detalle_local)
pd.testing.assert_frame_equal(resumen_reto, resumen_local)
pd.testing.assert_frame_equal(datos_reto, copia_reto)
for nombre, datos_caso in casos_practica.items():
    try:
        validar_libreria(datos_caso)
    except ValueError:
        if nombre == "correcto":
            raise
    else:
        if nombre != "correcto":
            raise AssertionError(f"La librería aceptó: {nombre}")
np.testing.assert_allclose(resumen_reto["importe_neto"].sum(), detalle_reto["importe_neto"].sum())
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(resumen_reto["producto"], resumen_reto["importe_neto"])
ax.set(title="Importe neto por producto — librería", xlabel="Producto", ylabel="Importe neto")
plt.show()
print(resumen_reto)
print("Reto verificado: imports, casos límite, equivalencia y entrada conservada.")


### Solución razonada

Las barras comparan importes agregados por producto. Las diferencias combinan número de operaciones, unidades, precio y descuentos; los productos se asignaron aleatoriamente y el gráfico no demuestra causalidad. Los módulos conservan las funciones, sus anotaciones y docstrings. El notebook se centra en visualizar e interpretar; las funciones del paquete pueden consumirse desde una CLI o pruebas.


# 28. Preguntas de reflexión


1. ¿Cuándo merece la pena crear una función?
2. ¿Qué diferencia existe entre parámetro y argumento?
3. ¿Por qué evitar demasiadas responsabilidades?
4. ¿Por qué son peligrosos los valores mutables por defecto?
5. ¿Qué aporta `return`?
6. ¿Los type hints evitan errores en runtime?
7. ¿Cuándo debemos validar manualmente?
8. ¿Qué diferencia existe entre documentación y validación?
9. ¿Qué debería contener una docstring profesional?
10. ¿Cuándo mover una función de `notebooks/` a `src/`?

## Respuestas orientativas

1. Crear una función merece la pena cuando existe una tarea con intención clara que necesita reutilizarse o probarse de forma aislada.
2. El parámetro aparece en la definición; el argumento es el valor que se pasa al llamar.
3. Separar responsabilidades permite probar y reutilizar cada etapa sin preparar todos los efectos y dependencias de una función monolítica.
4. Los valores mutables por defecto se crean una vez al definir la función y pueden acumular cambios entre llamadas; None permite crear un objeto nuevo por llamada.
5. return hace explícita la salida y permite componer funciones. Imprimir un resultado no lo devuelve; una función sin return devuelve None.
6. No. Los type hints orientan a lectores y herramientas de análisis estático; no hacen cumplir automáticamente los tipos en ejecución.
7. Debemos validar cuando las entradas pueden incumplir supuestos que afecten a la corrección: columnas, categorías, rangos, nulos y valores finitos.
8. La documentación explica el contrato; la validación lo comprueba al ejecutar. Ambas deben describir el mismo comportamiento.
9. Una docstring profesional explica propósito, parámetros, retorno, precondiciones, errores, efectos secundarios y ejemplos relevantes.
10. Conviene moverla cuando su lógica se estabiliza y debe reutilizarse, probarse o ejecutarse desde otros puntos de entrada. El notebook conserva exploración, gráficos e interpretación.
